# O–O–Anchored Local ISO Mass Density and Galaxy-Wide Integration  
### Cylindrical Milky Way Integration (McMillan 2017; baryonic & stellar templates)

This notebook implements the complete inference workflow used in the paper.  
It connects the **local interstellar-object (ISO) mass density**, constrained through:

1. a **magnitude-space population model** anchored to the Ōtautahi–Oxford (O–O) dynamical framework, and  
2. a **flux-through-a-sphere constraint** based on the **three observed ISOs**,

to derive a **Galaxy-integrated ISO mass** under Milky Way baryonic and stellar density templates (McMillan 2017).

---

## Workflow Overview

1. **Monte-Carlo modelling of the local ISO population (O–O model)**
   - Sample ISO population parameters:  
     $n_0$, $q_s$, albedo $p$, bulk density $\rho_{\rm bulk}$, faint-end cutoff $H_{\max}$.
   - Convert the magnitude distribution into a **local mass-density distribution**  
     $\rho_{\rm local}^{\rm OO}$ via the $H \to D \to m$ mapping.

2. **Rate-based constraint from 3 detections**
   - Use discovery dates, encounter speeds, and **individual detection radii**  
     to infer the posterior for the ISO arrival rate $\lambda$.  
   - Convert $\lambda$ into a **mass-density posterior**  
     $\rho_{\rm local}^{\rm rate}$.  
   - The method uses the **sum of cross-sections**  
     $$
     A_{\rm sum} = \sum_i \pi R_i^2,
     $$
     which is the physically correct form.

3. **Posterior fusion using KDE-weighted importance sampling**
   - Fit a KDE to $\log_{10}(\rho_{\rm local}^{\rm rate})$.  
   - Evaluate this KDE on the O–O samples to obtain a **likelihood weight**.  
   - Convert from log-space PDF to a linear-density PDF using  
     $$
     p(\rho) = \frac{p(\log_{10}\rho)}{\rho \ln 10}.
     $$
   - Clip extreme weights (avoids pathological dominance).  
   - Resample O–O draws proportional to these weights to obtain the  
     **joint posterior** $\rho_{\rm local}^{\rm joint}$.

4. **Galaxy-wide integration**
   - Construct **cylindrical Milky Way templates** using McMillan (2017).  
   - Assume ISO density follows the chosen template up to a normalisation at $(R_\odot, z=0)$.  
   - Integrate over $(R,z)$ to obtain the Galaxy-wide mass:
     $$
     M_{\rm ISO}
       = \int \rho_{\rm ISO}(R,z)\,2\pi R\, dR\, dz.
     $$

5. **Baryon-fraction inference**
   - Use  
     $$
     f_b = \frac{M_{\rm bary,known} + M_{\rm ISO}}{M_{\rm total}}
     $$
     to estimate how the ISO contribution modifies the Milky Way baryon fraction.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 12,
})

from galpy.potential import McMillan17
from astropy.constants import au, M_sun
from astropy import units as u

from scipy.stats import gamma, gaussian_kde
from datetime import datetime

from joblib import Parallel, delayed

## Global Configuration Parameters

In [ ]:
class CONFIG: 
    def __init__(self):
        
        # ---- Milky Way masses ----
        self.M_star = 5.4e10
        self.M_gas  = 1.2e10
        self.M_dust_fraction = 0.01
        self.M_dust = 0.01 * self.M_gas
        self.M_total = 1.17e12                           # Msun

        self.M_bary_known = self.M_star + self.M_gas + self.M_dust      # baryonic mass in Msun

        # ---- ISO population priors ----
        self.n0_centre = 0.2             # au^-3 (Do et al. 2018)
        self.n0_sigma_log10 = 0.5        # dex
        self.n0_min = 1e-2
        self.n0_max = 0.5

        self.q_s_min = 0.35
        self.q_s_max = 0.55

        self.albedo_min = 0.04
        self.albedo_max = 0.08

        self.rho_bulk_min = 500.0         # kg/m3
        self.rho_bulk_max = 2000.0

        self.H_min = 8.0
        self.H_max_centre = 22.1
        self.H_max_sigma = 0.3
        self.H_max_min = 21.0
        self.H_max_max = 23.0

        self.n_H_samples = 400

        # ---- ISO detections / rate model ----
        self.ISO_N_det = 3
        self.ISO_first_detection = "2017-10-19"
        self.ISO_last_detection  = "2025-07-01"

        self.ISO_detection_distance = [0.25, 2.98, 3.20]            # AU
        self.ISO_masses = [1e11, 2.6e11, 2e11]                      # kg
        self.ISO_velocities = [26.3e3, 32.0e3, 58.0e3]              # m/s

        self.ISO_detection_completeness = 1.0                       # lower bound

        # ---- KDE fusion tuning ----
        self.bw_factors = [0.7, 1.0, 1.3]                           # bandwidth factors to try
        self.weight_clip = 1e3

        # ---- McMillan grid ----
        self.R_min_kpc = 0.01
        self.R_max_kpc = 20.0
        self.n_R = 400

        self.z_min_kpc = -3.0
        self.z_max_kpc = +3.0
        self.n_z = 120

        # ---- Solar position ----
        self.R_sun_kpc = 8.2
        self.ro_kpc = 8.0                                           # galpy default scaling

        # ---- CI levels ----
        self.ci_low = 2.5
        self.ci_high = 97.5

        # Physical constants and unit conversions
        self.AU_m = au.to(u.m).value                                # 1 AU in meters
        self.AU3_m3 = self.AU_m**3                                  # 1 AU^3 in m^3
        self.Msun_kg = M_sun.to(u.kg).value
        self.kpc_to_m = (1 * u.kpc).to(u.m).value                   # 1 kpc in meters

        # Sample size for Monte Carlo
        self.N_samples = 1000000

        # Plotting 
        # reference baryon fractions (floats)
        self.f_b_MW_observed = self.M_bary_known / self.M_total     # from MW estimates
        self.f_b_expected = 0.08                                    # from external galaxies
        # vis
        self.fig_size_wide = (9,3)
        self.fontsize = 14
        self.hist_bins = 50

        self.color_scheme = {
            "oo_pdf":        "#4C72B0",
            "joint_pdf":     "#55A868",
            "joint_ci":      "#777777",
            "rate_interval": "#C44E52",
            "median":        "#222222",
            "ref":           "#222222",
            "mw":            "#222222",
            "expected":      "#8172B3",
        }


config = CONFIG()

# 1. ISO Population in Magnitude Space

This section defines priors on all population parameters and computes  
$\rho_{\rm local}$ through the mass integrated between  
$H_{\min} = 8$ and the sampled $H_{\max}$.

- Differential magnitude distribution:
  $$
  \frac{dN}{dH} \propto 10^{q_s H}.
  $$

- Diameter–magnitude relation:
  $$
  D(H,p) =
  1329\,{\rm km}\,
  \frac{10^{-H/5}}{\sqrt{p}}.
  $$

- Mass mapping:
  $$
  m(D,\rho_{\rm bulk}) = \frac{4\pi}{3}
     \left(\frac{D}{2}\right)^3 \rho_{\rm bulk}.
  $$

The Monte-Carlo loop produces the posterior  
$\rho_{\rm local}^{\rm OO}$.

### 1.1 Magnitude–Mass Mapping

In [ ]:
def mass_from_H(H, p, rho_bulk):
    """
    Mass of a body with absolute magnitude H, given albedo p and bulk density.

    Parameters
    ----------
    H : float or ndarray
        Absolute magnitude.
    p : float
        Geometric albedo.
    rho_bulk : float
        Bulk density [kg/m^3].

    Returns
    -------
    m : float or ndarray
        Mass [kg].
    """
    # Diameter in km (Masiero 2011 relation)
    D_km = 1329.0 * 10.0**(-H / 5.0) / np.sqrt(p)   # km

    # Radius in meters
    R_m = 0.5 * D_km * 1e3                          # m

    # Mass in kg
    m = (4.0 / 3.0) * np.pi * R_m**3 * rho_bulk
    return m

### 1.2 Local Mass Density for a Single Parameter Set

In [ ]:
def local_mass_density_single(
    n0_au3,
    q_s,
    p,
    rho_bulk,
    H_min,
    H_max,
    n_H_samples,
):
    """
    Compute local ISO mass density for a single parameter combination.

    Parameters
    ----------
    n0_au3 : float
        Cumulative number density n(H <= H_max) in au^-3.
    q_s : float
        Size-frequency slope in magnitude space.
    p : float
        Geometric albedo.
    rho_bulk : float
        Bulk density [kg/m^3].
    H_min, H_max : float
        Magnitude range over which the SFD is defined.
    n_H_samples : int
        Number of H grid points for numerical integration.

    Returns
    -------
    rho_local_m3 : float
        Local ISO mass density in kg/m^3.
    """
    # Magnitude grid
    H = np.linspace(H_min, H_max, n_H_samples)

    # Unnormalised shape of dN/dH
    shape = 10.0**(q_s * H)

    # Normalisation constant A so that integral = n0_au3
    integral_shape = np.trapezoid(shape, H)
    if integral_shape <= 0:
        return 0.0

    A = n0_au3 / integral_shape

    # Differential number density [au^-3 mag^-1]
    dN_dH = A * shape

    # Mass per object [kg]
    m_H = mass_from_H(H, p=p, rho_bulk=rho_bulk)

    # Differential mass density [kg au^-3 mag^-1]
    dM_dH = m_H * dN_dH

    # Integrate over H -> kg au^-3
    rho_local_au3 = np.trapezoid(dM_dH, H)

    # Convert to kg m^-3
    rho_local_m3 = rho_local_au3 / config.AU3_m3
    return rho_local_m3

### 1.3 Parameter Priors and O–O Monte-Carlo

In [ ]:
def sample_parameters_OO(N_samples, rng=None):
    """
    Draw Monte-Carlo samples of the ISO population parameters
    under the O–O-motivated priors.

    Returns
    -------
    n0_au3   : ndarray, shape (N_samples,)
        Local cumulative number density [au^-3].
    q_s      : ndarray
        SFD slope in H-space.
    p        : ndarray
        Geometric albedo.
    rho_bulk : ndarray
        Bulk density [kg m^-3].
    H_max    : ndarray
        Faint-end cutoff in H.
    """
    if rng is None:
        rng = np.random.default_rng()

    # 1) n0 ~ lognormal centred on 0.2 au^-3, truncated
    log10_n0_centre = np.log10(config.n0_centre)   # Do et al. 2018
    sigma_log10_n0 = config.n0_sigma_log10         # ~0.5 dex uncertainty

    log10_n0 = rng.normal(log10_n0_centre, sigma_log10_n0, size=N_samples)
    n0_au3 = 10.0**log10_n0
    n0_au3 = np.clip(n0_au3, config.n0_min, config.n0_max)  # [0.01, 0.5] au^-3

    # 2) q_s slope
    q_s = rng.uniform(config.q_s_min, config.q_s_max, size=N_samples)

    # 3) Geometric albedo
    p = rng.uniform(config.albedo_min, config.albedo_max, size=N_samples)

    # 4) Bulk density
    rho_bulk = rng.uniform(config.rho_bulk_min, config.rho_bulk_max, size=N_samples)

    # 5) H_max
    H_max = rng.normal(config.H_max_centre, config.H_max_sigma, size=N_samples)
    H_max = np.clip(H_max, config.H_max_min, config.H_max_max)

    return n0_au3, q_s, p, rho_bulk, H_max


def monte_carlo_rho_OO(rng=None):
    """
    Monte-Carlo sampling of the local ISO mass density under the O–O priors.

    Parameters
    ----------
    rng : np.random.Generator or None
        Optional random generator.

    Returns
    -------
    rho_local_samples : ndarray
        Array of shape (N_samples,) with rho_local in kg/m^3.
    summary : dict
        Dictionary with keys "rho_low", "rho_best", "rho_high".
    """
    if rng is None:
        rng = np.random.default_rng()

    n0_au3, q_s, p, rho_bulk, H_max = sample_parameters_OO(config.N_samples, rng=rng)

    if isinstance(config.N_samples, int):
        H_min = config.H_min
    else:
        raise ValueError("config.N_samples must be an integer")

    rho_local_samples = np.empty(config.N_samples, dtype=float)

    for i in range(config.N_samples):
        rho_local_samples[i] = local_mass_density_single(
            n0_au3=n0_au3[i],
            q_s=q_s[i],
            p=p[i],
            rho_bulk=rho_bulk[i],
            H_min=H_min,
            H_max=H_max[i],
            n_H_samples=config.n_H_samples,
        )

    rho_low  = np.percentile(rho_local_samples, config.ci_low)
    rho_best = np.percentile(rho_local_samples, 50.0)
    rho_high = np.percentile(rho_local_samples, config.ci_high)

    summary = {
        "rho_low": rho_low,
        "rho_best": rho_best,
        "rho_high": rho_high,
    }

    return rho_local_samples, summary

# 2. Galactic Templates (McMillan 2017)

Cylindrical $(R,z)$ grids are constructed and:

- stellar,
- bulge,  
- gas  

densities are evaluated using `galpy.potential.McMillan17`.

We use templates only **up to proportionality**, normalised at the Solar location:

$$
\rho_{\rm ISO}(R,z) =
   \rho_{\rm local}
   \frac{\rho_{\rm template}(R,z)}{\rho_{\rm template,\odot}}.
$$

In [ ]:
# ============================================================
# 6. Galactic templates from McMillan (2017) – baryonic & stellar
# ============================================================

def build_templates_McMillan():
    """
    Build cylindrical (R,z) grids and corresponding 3D density templates
    using McMillan (2017):

      - Stellar template:  rho_star(R,z)
      - Baryonic template: rho_bary(R,z) = star + bulge + gas

    All densities are in arbitrary units consistent with McMillan17.
    We keep them *relative* – only ratios rho_T / rho_T_sun matter.

    Returns
    -------
    R_kpc, z_kpc : 1D arrays
    rho_bary_grid, rho_bary_sun : 2D array, float
    rho_star_grid, rho_star_sun : 2D array, float
    """

    # Physical grid in kpc for the ISO integration
    R_kpc = np.linspace(config.R_min_kpc, config.R_max_kpc, config.n_R)    # 0.01–20 kpc
    z_kpc = np.linspace(config.z_min_kpc, config.z_max_kpc, config.n_z)     # -3–+3 kpc

    # galpy's McMillan17 uses dimensionless radii: R/ro, z/ro
    R_dim = R_kpc / config.ro_kpc
    z_dim = z_kpc / config.ro_kpc
    RR_dim, ZZ_dim = np.meshgrid(R_dim, z_dim, indexing="ij")

    # Vectorized density helpers (in McMillan's units; relative shapes only)
    stellar = np.vectorize(McMillan17.stellar_dens)
    bulge   = np.vectorize(McMillan17.bulge_dens)
    gas     = np.vectorize(McMillan17.gas_dens)

    rho_star_grid  = stellar(RR_dim, ZZ_dim)
    rho_bulge_grid = bulge(RR_dim, ZZ_dim)
    rho_gas_grid   = gas(RR_dim, ZZ_dim)

    # Baryonic template = stars + bulge + gas
    rho_bary_grid  = rho_star_grid + rho_bulge_grid + rho_gas_grid

    # Local normalisations at the Sun (R⊙ = 8.2 kpc, z = 0)
    R_sun_dim = config.R_sun_kpc / config.ro_kpc

    rho_star_sun  = stellar(R_sun_dim, 0.0)
    rho_bulge_sun = bulge(R_sun_dim, 0.0)
    rho_gas_sun   = gas(R_sun_dim, 0.0)
    rho_bary_sun  = rho_star_sun + rho_bulge_sun + rho_gas_sun

    templates = {
        "stellar":  (rho_star_grid, rho_star_sun),
        "baryonic": (rho_bary_grid, rho_bary_sun)
    }

    return R_kpc, z_kpc, templates

# 3. ISO Mass Integration

Total Galaxy-wide ISO mass is:

$$
M_{\rm ISO} =
\int_{R_{\min}}^{R_{\max}}
\int_{-z_{\max}}^{+z_{\max}}
\rho_{\rm ISO}(R,z)\,2\pi R\, dR\, dz.
$$

After converting kpc to meters, this is evaluated numerically on the grid.

For every posterior sample of $\rho_{\rm local}$, we obtain a sample of $M_{\rm ISO}$.

In [ ]:
def integrate_ISO_mass_cylindrical(
    rho_local_m3,
    rho_template_grid,
    rho_template_sun,
    R_kpc,
    z_kpc,
):
    """
    Compute total ISO mass for a given local true mass density at (R_sun, 0),
    assuming ISO density follows the provided baryonic template.

    Parameters
    ----------
    rho_local_m3 : float
        Local true ISO mass density at (R_sun, 0) [kg/m^3].
    rho_template_grid : ndarray
        Template density values on the (R,z) grid.
    rho_template_sun : float
        Template density value at the Sun.
    R_kpc : ndarray
        Radial grid [kpc].
    z_kpc : ndarray
        Vertical grid [kpc].

    Returns
    -------
    M_total_ISO_Msun : float
        Total ISO mass in units of solar masses.
    """
    # Normalised ISO density on the grid
    rho_IS = rho_local_m3 * (rho_template_grid / rho_template_sun)

    # Cylindrical volume element dV = 2π R dR dz
    R_m = R_kpc * config.kpc_to_m
    dR_m = np.gradient(R_kpc) * config.kpc_to_m
    dz_m = np.gradient(z_kpc) * config.kpc_to_m

    dR_2d = dR_m[:, None]
    dz_2d = dz_m[None, :]

    dV = 2.0 * np.pi * R_m[:, None] * dR_2d * dz_2d  # m^3

    M_total_ISO_kg = np.sum(rho_IS * dV)
    M_total_ISO_Msun = M_total_ISO_kg / config.Msun_kg

    return M_total_ISO_Msun


def MISO_from_rho(
    rho_samples,
    rho_template_grid,
    rho_template_sun,
    R_kpc,
    z_kpc,
):
    """
    Compute Galaxy-integrated ISO mass for many rho_local samples.

    Parameters
    ----------
    rho_samples : array_like
        1D array of local ISO mass densities (kg/m^3).
    rho_template_grid : ndarray
        The Galactic density template grid (e.g., rho_bary_grid).
    rho_template_sun : float
        Density of the template at the Sun.
    R_kpc, z_kpc : ndarray
        The cylindrical grid.

    Returns
    -------
    M_ISO : ndarray
        Array of integrated ISO masses in Msun for each rho value.
    """
    rho_samples = np.asarray(rho_samples)
    M = np.empty_like(rho_samples, dtype=float)

    for i, rho_local in enumerate(rho_samples):
        M[i] = integrate_ISO_mass_cylindrical(
            rho_local_m3=rho_local,
            rho_template_grid=rho_template_grid,
            rho_template_sun=rho_template_sun,
            R_kpc=R_kpc,
            z_kpc=z_kpc,
        )

    return M

## 4. Rate-Based Constraint from the Three Detected ISOs

The local ISO mass density is independently constrained using the observed
detection rate of interstellar objects and their individual discovery
geometries.

### 4.1 Arrival-rate posterior

Assuming Poisson statistics for detections over the observation baseline
$T_{\rm obs}$, the posterior distribution for the arrival rate $\lambda$ is

$$
\lambda \sim \Gamma\!\left(N_{\rm det}+1,\; \text{scale}=\frac{1}{T_{\rm obs}}\right),
$$

where $N_{\rm det}=3$ is the number of detected ISOs.

### 4.2 Effective geometric exposure

Each detected ISO contributes an individual geometric detection
cross-section $\pi R_i^2$, where $R_i$ is the heliocentric distance at first
secure detection.

To convert an arrival rate into a local number density, we define the
effective exposure as the sum of cross-section–velocity products,

$$
A_v \equiv \sum_{i=1}^{N_{\rm det}} \pi R_i^2\, v_{\infty,i},
$$

where $v_{\infty,i}$ is the asymptotic encounter speed of object $i$ relative
to the Solar System.

This quantity has dimensions of volume per unit time and directly converts
the arrival rate into a cumulative number density.

### 4.3 Number-density inference

The local cumulative ISO number density inferred from the rate constraint is

$$
n_0^{\rm rate} = \frac{\lambda}{C\,A_v},
$$

where $C$ denotes the detection completeness. In the baseline analysis we
adopt $C=1$, making the resulting constraint conservative.

### 4.4 Mean ISO mass

The characteristic ISO mass is estimated by bootstrap resampling of the
three observed objects (1I/‘Oumuamua, 2I/Borisov, and 3I/ATLAS), yielding a
posterior distribution for the mean mass $\langle m \rangle$.

### 4.5 Local mass-density posterior

The rate-based posterior for the local ISO mass density is then obtained as

$$
\rho_{\rm local}^{\rm rate}
  = \langle m \rangle\, n_0^{\rm rate}.
$$

This posterior is intrinsically skewed and spans several orders of
magnitude, motivating the use of logarithmic kernel density estimation in
the subsequent posterior-fusion step.


In [ ]:
def compute_rho_rate_samples():
    """
    Compute the rate-based posterior over rho_local using the corrected
    cross-section sum A_sum = sum_i (π R_i^2), avoiding bimodal artifacts.
    """

    # --- 1. Compute time baseline T_obs in seconds ---
    t0 = datetime.fromisoformat(config.ISO_first_detection)
    t1 = datetime.fromisoformat(config.ISO_last_detection)
    T_obs = (t1 - t0).days * 86400.0

    # --- 2. Compute SUM of cross-sections (correct physics) ---
    R_arr_m = np.array(config.ISO_detection_distance) * config.AU_m     # convert AU → m
    A_list  = np.pi * R_arr_m**2              # list of π R_i^2
    A_sum   = np.sum(A_list)                  # total effective area

    # --- 3. Posterior samples of λ (correct Gamma posterior) ---
    # Posterior: lambda ~ Gamma(N_det + 1, rate=T_obs)
    lam_samples = gamma.rvs(a=config.ISO_N_det + 1, scale=1.0 / T_obs, size=config.N_samples)

    # --- 4. Bootstrap the mean ISO mass ---
    masses_arr = np.array(config.ISO_masses)
    idx = np.random.randint(0, len(masses_arr), (config.N_samples, len(masses_arr)))
    mean_mass_samples = masses_arr[idx].mean(axis=1)

    # --- 5. Number density (m^-3) ---
    
    v_inf_m_s = np.array(config.ISO_velocities)    # m/s
    A_v_sum = np.sum(A_list * v_inf_m_s)

    n0_samples_m3 = lam_samples / (config.ISO_detection_completeness * A_v_sum)

    # --- 6. Rate-based mass density ---
    rho_rate_samples = mean_mass_samples * n0_samples_m3  # kg/m^3

    return rho_rate_samples


# 5. KDE-Weighted Fusion of O–O and Rate Constraints

We fuse the two independent constraints by treating the rate-based PDF as a  
likelihood applied to the O–O prior.

Steps:

1. Fit a KDE to $\log_{10}(\rho_{\rm local}^{\rm rate})$.
2. Evaluate it at $\log_{10}(\rho_{\rm local}^{\rm OO})$.
3. Convert:
   $$
   p(\rho) = \frac{p(\log_{10}\rho)}{\rho \ln 10}.
   $$
4. Clip extreme weights.
5. Resample O–O draws proportionally to obtain  
   $$
   \rho_{\rm local}^{\rm joint}.
   $$

In [ ]:
def build_rate_kde_log10(rho_rate_samples, bw_factor):
    """
    Build a KDE over log10(rho_local) from the rate-based samples.

    Returns a gaussian_kde object that takes log10(rho) as input.
    """
    rho_rate_samples = np.asarray(rho_rate_samples)
    # Work only with strictly positive values
    rho_rate_samples = rho_rate_samples[rho_rate_samples > 0]
    log_rho_rate = np.log10(rho_rate_samples)

    # Bandwidth: factor * Scott's rule
    kde = gaussian_kde(log_rho_rate, bw_method="scott")
    if bw_factor != 1.0:
        kde.set_bandwidth(bw_method=kde.factor * bw_factor)
    return kde


def fuse_OO_and_rate_posteriors_KDE(
    rho_OO_samples,
    rho_rate_samples,
    bw_factor,
    rng=None
):
    """
    Fuse O–O and rate-based posteriors into a joint posterior over rho_local
    by treating the rate-based posterior as a likelihood and doing
    importance resampling of the O–O samples.

    Steps:
      1. Fit a KDE to log10(rho_rate) -> p_log(log10 rho | rate).
      2. Evaluate this KDE at log10(rho_OO).
      3. Convert to density in rho with Jacobian 1 / (rho ln 10).
      4. Clip extreme weights to avoid domination by a few points.
      5. Resample rho_OO with probabilities ∝ weight to obtain
         rho_kde_samples.

    Parameters
    ----------
    rho_OO_samples : array_like
        O–O Monte-Carlo samples of rho_local [kg/m^3].
    rho_rate_samples : array_like
        Rate-based posterior samples of rho_local [kg/m^3].
    bw_factor : float
        Multiplicative factor on Scott's bandwidth for KDE smoothing.
        >1.0 => smoother KDE; <1.0 => more peaky.
    weight_clip : float
        Maximum allowed ratio of weights relative to the median.
        (Weights are normalized after clipping.)
    rng : np.random.Generator or None
        RNG for resampling.

    Returns
    -------
    rho_kde_samples: ndarray
        Joint posterior samples of rho_local [kg/m^3].
    weights : ndarray
        Normalized importance weights applied to each O–O sample.
    """
    rho_OO_samples = np.asarray(rho_OO_samples)
    rho_rate_samples = np.asarray(rho_rate_samples)

    N_joint = len(rho_OO_samples)
    
    if rng is None:
        rng = np.random.default_rng()

    # --- 1. Build KDE on log10(rho_rate) ---
    kde_rate = build_rate_kde_log10(rho_rate_samples=rho_rate_samples, bw_factor=bw_factor)

    # --- 2. Evaluate KDE at log10(rho_OO) ---
    # Work only with positive O–O samples
    mask_pos = rho_OO_samples > 0
    if not np.any(mask_pos):
        raise RuntimeError("All O–O rho samples are non-positive; cannot fuse posteriors.")

    rho_OO_pos = rho_OO_samples[mask_pos]
    log_rho_OO_pos = np.log10(rho_OO_pos)

    # p_log = p(log10 rho | rate)
    p_log = kde_rate(log_rho_OO_pos)  # always >= 0

    # --- 3. Convert to density in rho: p(rho) = p_log / (rho ln 10) ---
    # This is the Jacobian for the log10 transform.
    p_rho = p_log / (rho_OO_pos * np.log(10.0))

    # Replace NaN / inf / negative with zero
    p_rho = np.where(np.isfinite(p_rho) & (p_rho > 0), p_rho, 0.0)

    if p_rho.sum() <= 0:
        raise RuntimeError("Rate likelihood vanished everywhere on O–O support; fusion failed.")

    # --- 4. Clip extreme weights to avoid pathological dominance ---
    # Start from unnormalized weights = p_rho
    weights = p_rho.copy()

    # Normalize to have median ~1 before clipping
    positive = weights > 0
    if np.any(positive):
        w_med = np.median(weights[positive])
        if w_med <= 0:
            w_med = np.mean(weights[positive])
        if w_med > 0:
            weights /= w_med

    # Clip to [1/weight_clip, weight_clip]
    weights = np.clip(weights, 1.0 / config.weight_clip, config.weight_clip)

    # Re-normalize
    weights /= weights.sum()

    # --- 5. Resample rho_OO according to weights to form joint posterior ---
    idx = rng.choice(np.where(mask_pos)[0], size=N_joint, replace=True, p=weights)
    rho_kde_samples = rho_OO_samples[idx]

    # Construct full-length weights array aligned to rho_OO_samples
    full_weights = np.zeros_like(rho_OO_samples, dtype=float)
    full_weights[mask_pos] = weights

    return rho_kde_samples, full_weights


# 6. Running the Full Joint Inference

This wrapper performs:

- O–O Monte-Carlo,
- rate posterior,
- KDE-weighted fusion,
- Galaxy integration for each template,
- baryon-fraction inference.

All results are stored in a unified dictionary.

In [ ]:
def _integrate_joint_for_template(task):
    """
    Worker for joint ISO mass integration.
    Windows- and joblib-safe (top-level).
    """
    template_name, rho_kde_samples, rho_grid, rho_sun, R_kpc, z_kpc = task

    MISO_kde = MISO_from_rho(
        rho_kde_samples, rho_grid, rho_sun, R_kpc, z_kpc
    )

    return template_name, MISO_kde


def run_joint_OO_plus_rate_analysis(parallelize=False):
    """
    Run the joint O–O + rate-based analysis, including Galactic integration,
    using KDE-weighted posterior fusion, for multiple KDE bandwidth factors.

    Returns
    -------
    results : dict
        Structure:
          results["templates"][template_name] -> template-level outputs (bw-independent)
          results["bw"][bw_factor][template_name] -> bw-dependent joint outputs
          results["meta"] -> shared inputs/summaries
    """

    def summarize_rho(samples):
        return {
            "rho_low":  np.percentile(samples, config.ci_low),
            "rho_best": np.percentile(samples, 50.0),
            "rho_high": np.percentile(samples, config.ci_high),
        }

    def summarize_fb(M_iso):
        f_b = (config.M_bary_known + M_iso) / config.M_total
        f_ISO = M_iso / config.M_total
        summary = {
            "f_ISO_low":  np.percentile(f_ISO, config.ci_low),
            "f_ISO_best": np.percentile(f_ISO, 50.0),
            "f_ISO_high": np.percentile(f_ISO, config.ci_high),
            "fb_low":  np.percentile(f_b, config.ci_low),
            "fb_best": np.percentile(f_b, 50.0),
            "fb_high": np.percentile(f_b, config.ci_high),
        }
        return f_b, summary

    rng = np.random.default_rng()

    # --- 1) Sample local rho posteriors (bw-independent) ---
    rho_OO_samples, _ = monte_carlo_rho_OO(rng=rng)
    rho_rate_samples = compute_rho_rate_samples()

    rho_OO_summary = summarize_rho(rho_OO_samples)
    rho_rate_summary = summarize_rho(rho_rate_samples)

    # --- 2) Build Galactic templates ONCE (bw-independent) ---
    R_kpc, z_kpc, templates = build_templates_McMillan()

    # --- 3) Precompute OO and rate Galactic masses ONCE per template (bw-independent) ---
    templates_out = {}
    for template_name, (rho_grid, rho_sun) in templates.items():
        MISO_OO = MISO_from_rho(rho_OO_samples, rho_grid, rho_sun, R_kpc, z_kpc)
        MISO_rate = MISO_from_rho(rho_rate_samples, rho_grid, rho_sun, R_kpc, z_kpc)

        f_b_OO_samples, f_b_OO_summary = summarize_fb(MISO_OO)
        f_b_rate_samples, f_b_rate_summary = summarize_fb(MISO_rate)

        templates_out[template_name] = {
            # template definition (small, useful for debugging)
            "rho_template_sun": rho_sun,

            # bw-independent local posteriors (store once here to avoid duplication)
            "rho_OO_samples": rho_OO_samples,
            "rho_OO_summary": rho_OO_summary,
            "rho_rate_samples": rho_rate_samples,
            "rho_rate_summary": rho_rate_summary,

            # bw-independent integrated masses + fb
            "MISO_OO": MISO_OO,
            "MISO_rate": MISO_rate,
            "f_b_OO_samples": f_b_OO_samples,
            "f_b_rate_samples": f_b_rate_samples,
            "f_b_OO_summary": f_b_OO_summary,
            "f_b_rate_summary": f_b_rate_summary,
        }

    # --- 4) For each bw_factor, fuse once, then integrate JOINT per template ---
    bw_out = {}
    for bw_factor in config.bw_factors:
        rho_kde_samples, fusion_weights = fuse_OO_and_rate_posteriors_KDE(
            rho_OO_samples=rho_OO_samples,
            rho_rate_samples=rho_rate_samples,
            bw_factor=bw_factor,
            rng=rng,
        )
        rho_joint_summary = summarize_rho(rho_kde_samples)

        bw_key = f"bw_{bw_factor}"
        bw_out[bw_key] = {
            "rho_kde_samples": rho_kde_samples,
            "rho_joint_summary": rho_joint_summary,
            "fusion_weights": fusion_weights,
        }

        if parallelize:
            tasks = [
                (
                    template_name,
                    rho_kde_samples,
                    rho_grid,
                    rho_sun,
                    R_kpc,
                    z_kpc,
                )
                for template_name, (rho_grid, rho_sun) in templates.items()
            ]

            results_parallel = Parallel(n_jobs=-1, backend="loky")(
                delayed(_integrate_joint_for_template)(task)
                for task in tasks
            )

            for template_name, MISO_kde in results_parallel:
                f_b_kde_samples, f_b_kde_summary = summarize_fb(MISO_kde)

                bw_out[bw_key][template_name] = {
                    "MISO_kde": MISO_kde,
                    "f_b_kde_samples": f_b_kde_samples,
                    "f_b_kde_summary": f_b_kde_summary,
                }
        else:
            # integrate joint per template (this is the only expensive bw-dependent part)
            for template_name, (rho_grid, rho_sun) in templates.items():
                MISO_kde = MISO_from_rho(rho_kde_samples, rho_grid, rho_sun, R_kpc, z_kpc)
                f_b_kde_samples, f_b_kde_summary = summarize_fb(MISO_kde)



                bw_out[bw_key][template_name] = {
                    "MISO_kde": MISO_kde,
                    "f_b_kde_samples": f_b_kde_samples,
                    "f_b_kde_summary": f_b_kde_summary,
                }

    # --- 5) Package outputs compactly ---
    results = {
        "meta": {
            "ci_low": config.ci_low,
            "ci_high": config.ci_high,
            "M_bary_known": config.M_bary_known,
            "M_total": config.M_total,
            "R_grid_kpc": R_kpc,
            "z_grid_kpc": z_kpc,
            "bw_factors": list(config.bw_factors),
        },
        "templates": templates_out,
        "bw": bw_out,
    }

    return results


In [ ]:
results = run_joint_OO_plus_rate_analysis(parallelize=True)

# 7. Plotting

## Post-processing: figures, tables, and numerical summaries

This section contains post-processing utilities that operate on the output
dictionary returned by `run_joint_OO_plus_rate_analysis()`.

No additional modelling or inference is performed below.
All functions in this section:

- take the already-sampled posterior distributions as input,
- summarise them in graphical or tabular form,
- expose numerical values required to populate the Results section of the paper.

Unless stated otherwise, figures and tables are produced for the default
KDE bandwidth (`bw_factor = 1.0`), corresponding to Scott’s rule.
Alternative bandwidths are used only for robustness checks.


In [ ]:
def _c(key: str, fallback: str) -> str:
    """Colour lookup helper with fallback to existing scheme."""
    return config.color_scheme.get(key, config.color_scheme.get(fallback))

def format_sci(df, columns, precision=2):
    """CSV output rounding helper."""
    for c in columns:
        df[c] = df[c].map(lambda x: f"{x:.{precision}e}")
    return df

## Figure: Local ISO mass density — individual channels and joint posterior

This figure compares the three posterior distributions for the local ISO mass
density $\rho_{\rm local}$:

1. The magnitude-space population-model posterior (O–O model),
2. The joint posterior obtained after KDE-weighted fusion with the
   detection-rate constraint,
3. The central credible interval of the joint posterior.

The purpose of this plot is to visualise how the two independent inference
channels combine into a single posterior distribution.
All quantities shown here correspond directly to
Eqs.~(\\ref{eq:rho_OO_results}), (\\ref{eq:rho_rate_results}),
and (\\ref{eq:rho_joint_results}) in the Results section.

Expected output:
- A log–log histogram of $\rho_{\rm local}$,
- Shaded region marking the joint 95% credible interval.


In [ ]:
def plot_rho_local_channels(results, bw_factor=1.0, template="baryonic", figure_name="rho_local_channels"):
    """
    Compare rho_local posteriors:
      - O–O posterior (histogram)
      - joint posterior (histogram or KDE-like via histogram)
      - joint 95% CI shading
      - rate-only 95% CI shown as a horizontal interval bar

    Uses config.color_scheme for all colours.
    """
    bw_key = f"bw_{bw_factor}"
    tpl = results["templates"][template]
    bw = results["bw"][bw_key]

    rho_OO = tpl["rho_OO_samples"]
    rho_rate = tpl["rho_rate_samples"]
    rho_joint = bw["rho_kde_samples"]

    # Interval summaries
    rho_joint_low = bw["rho_joint_summary"]["rho_low"]
    rho_joint_med = bw["rho_joint_summary"]["rho_best"]
    rho_joint_high = bw["rho_joint_summary"]["rho_high"]

    rho_rate_low = tpl["rho_rate_summary"]["rho_low"]
    rho_rate_high = tpl["rho_rate_summary"]["rho_high"]

    xmin = min(rho_OO.min(), rho_rate.min(), rho_joint.min()) * 0.8
    xmax = max(rho_OO.max(), rho_rate.max(), rho_joint.max()) * 1.2
    bins = np.logspace(np.log10(xmin), np.log10(xmax), 200)

    plt.figure(figsize=config.fig_size_wide)

    plt.hist(
        rho_OO, bins=bins, density=True,
        color=_c("oo_pdf", "hist"),
        alpha=0.45,
        label=r"Ō—O posterior $P_{\rm OO}(\rho_{\rm local})$",
    )

    # KDE-weighted fused posterior (this is what you call “joint” in the method)
    plt.hist(
        rho_joint, bins=bins, density=True,
        color=_c("joint_pdf", "ci"),
        alpha=0.35,
        label=r"KDE-weighted posterior $P_{\rm KDE}(\rho_{\rm local})$",
    )

    # CI shading derived from KDE-weighted posterior
    plt.axvspan(
        rho_joint_low, rho_joint_high,
        color=_c("joint_ci", "interval"),
        alpha=0.18,
        label=r"95\% credible interval of $P_{\rm KDE}$",
    )

    plt.axvline(
        rho_joint_med,
        color=_c("median", "median"),
        linestyle="-",
        linewidth=2,
        label=r"Median of $P_{\rm KDE}$",
    )


    plt.xscale("log")
    plt.yscale("log")

    # Rate-only CI as horizontal bar near bottom
    y_min, y_max = plt.ylim()
    y_bar = y_min * 1.5
    # Rate-only interval
    plt.hlines(
        y_bar,
        rho_rate_low,
        rho_rate_high,
        colors=_c("rate_interval", "rate"),
        linewidth=4,
        label=r"95% interval (rate-only)",
    )
    plt.xlim(4e-30, xmax)
    plt.ylim(y_min * 0.5, y_max)

    plt.xlabel(r"Local ISO mass density $\rho_{\rm local}$ [kg m$^{-3}$]")
    plt.ylabel("Probability density")
    plt.title("")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
    f"results/{figure_name}.pdf",
    format="pdf",
    bbox_inches="tight",
    )
    plt.show()
    plt.close()


In [ ]:
plot_rho_local_channels(results)

## Figure: KDE-weighted posterior and posterior reshaping

This figure illustrates the effect of the detection-rate likelihood on the
magnitude-space population posterior using kernel density estimates (KDEs)
in $\log_{10}(\rho_{\rm local})$.

Two panels are produced:

1. A comparison of the KDE-smoothed O–O posterior and the KDE-smoothed
   joint posterior,
2. The ratio of the two KDEs, which quantifies how the rate-based likelihood
   reweights different regions of $\rho_{\rm local}$.

This visualisation corresponds to Eq.~(\\ref{eq:kde_ratio}) and is used
to document the internal consistency of the posterior fusion procedure.

Expected output:
- KDE curves in log–log space,
- A ratio curve centred around unity where the likelihood is neutral.

In [ ]:
def plot_kde_and_ratio(results, bw_factor=1.0, template="baryonic", figure_name="rho_local_kde_ratio"):
    """
    Panel A: KDE in log10(rho) for O–O and joint
    Panel B: ratio KDE_joint / KDE_OO (clipped to well-supported region)

    Uses config.color_scheme for all colours.
    """
    from scipy.stats import gaussian_kde

    bw_key = f"bw_{bw_factor}"
    tpl = results["templates"][template]
    bw = results["bw"][bw_key]

    rho_OO = tpl["rho_OO_samples"]
    rho_joint = bw["rho_kde_samples"]

    # Joint CI
    rho_joint_low = bw["rho_joint_summary"]["rho_low"]
    rho_joint_med = bw["rho_joint_summary"]["rho_best"]
    rho_joint_high = bw["rho_joint_summary"]["rho_high"]

    log_OO = np.log10(rho_OO[rho_OO > 0])
    log_joint = np.log10(rho_joint[rho_joint > 0])

    xs = np.linspace(
        min(log_OO.min(), log_joint.min()),
        max(log_OO.max(), log_joint.max()),
        400,
    )
    rho_x = 10 ** xs

    kde_OO = gaussian_kde(log_OO)(xs)
    kde_joint = gaussian_kde(log_joint)(xs)

    # ============================================================
    # Panel A: KDE overlay (CLIPPED + colour-scheme aligned)
    # ============================================================

    rho_all = np.concatenate([rho_OO, rho_joint])
    rho_lo_A, rho_hi_A = np.percentile(rho_all, [0.25, 99.75])

    support_A = (
        (rho_x >= rho_lo_A) &
        (rho_x <= rho_hi_A) &
        (
            (kde_OO > 1e-6 * kde_OO.max()) |
            (kde_joint > 1e-6 * kde_joint.max())
        )
    )

    plt.figure(figsize=config.fig_size_wide)

    plt.plot(
        rho_x[support_A],
        kde_OO[support_A],
        color=_c("hist", "hist"),
        linewidth=2,
        alpha=0.85,
        label="O--O KDE",
    )

    plt.plot(
        rho_x[support_A],
        kde_joint[support_A],
        color=_c("joint", "joint"),
        linewidth=2,
        alpha=0.95,
        label="Joint KDE",
    )

    plt.axvspan(
        rho_joint_low,
        rho_joint_high,
        color=_c("interval", "ci"),
        alpha=0.15,
        label="95% credible interval (joint)",
    )

    plt.axvline(
        rho_joint_med,
        color=_c("median", "median"),
        linewidth=2,
    )

    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel(r"Local ISO mass density $\rho_{\rm local}$ [kg m$^{-3}$]")
    plt.ylabel("KDE density (arbitrary units)")
    plt.title("KDE Representation of Local ISO Mass Density Posteriors")
    plt.legend()
    plt.tight_layout()
    plt.show()


    # ============================================================
    # Panel B: KDE ratio (CLIPPED)
    # ============================================================

    # --- support limits: central 99% of joint posterior ---
    rho_lo_clip, rho_hi_clip = np.percentile(rho_joint, [0.5, 99.5])

    support_mask = (
        (rho_x >= rho_lo_clip) &
        (rho_x <= rho_hi_clip) &
        (kde_OO > 1e-6 * np.max(kde_OO))
    )

    rho_x_clip = rho_x[support_mask]
    ratio_clip = (kde_joint / kde_OO)[support_mask]

    plt.figure(figsize=config.fig_size_wide)
    plt.plot(
        rho_x_clip,
        ratio_clip,
        color=_c("interval", "interval"),
        linewidth=2,
        label=r"${\rm KDE}_{\rm joint}/{\rm KDE}_{\rm OO}$",
    )
    plt.axhline(
        1.0,
        color=_c("ref", "median"),
        linestyle="--",
        linewidth=1.5,
        alpha=0.8,
    )

    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel(r"Local ISO mass density $\rho_{\rm local}$ [kg m$^{-3}$]")
    plt.ylabel(r"$\mathcal{R}(\rho_{\rm local})$")
    plt.title("Posterior Reshaping Induced by the Rate Likelihood")
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        f"results/{figure_name}.pdf",
        format="pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close()


In [ ]:
plot_kde_and_ratio(results)

## Figure: Posterior distribution of the baryon fraction

This figure shows the posterior distribution of the baryon fraction $f_b$
after including the ISO contribution inferred from the joint posterior
$P_{\rm joint}(\rho_{\rm local})$.

Each sample of $\rho_{\rm local}$ is propagated through the Galactic
integration and converted to $f_b$ using the fixed Milky Way baryonic
and total masses.

The plotted distribution corresponds directly to
Eq.~(\\ref{eq:fb_results}) in the Results section.

Expected output:
- A log–log histogram of $f_b$,
- Vertical lines indicating the median and central credible interval.


In [ ]:
def plot_fb_posterior(results, bw_factor=1.0, template="baryonic", figure_name="fb_posterior"):
    """
    Posterior distribution of f_b for a given template under joint posterior.

    Uses config.color_scheme for histogram + interval + reference lines.
    """
    bw_key = f"bw_{bw_factor}"
    for template in ["baryonic", "stellar"]:
        fb = results["bw"][bw_key][template]["f_b_kde_samples"]
        summ = results["bw"][bw_key][template]["f_b_kde_summary"]

        # Reference values (if you have them in config, use those; else compute MW inventory fraction)
        f_b_MW_inventory = results["meta"]["M_bary_known"] / results["meta"]["M_total"]
        f_b_expected = getattr(config, "f_b_expected", None)  # optional

        xmin = max(fb.min() * 0.9, 1e-6)
        xmax = fb.max() * 1.1
        bins = np.logspace(np.log10(xmin), np.log10(xmax), 200)

        plt.figure(figsize=config.fig_size_wide)

        plt.hist(
            fb, bins=bins, density=True,
            color=_c("interval", "interval"),
            alpha=0.55,
            label=r"Posterior $P(f_b)$ (joint)",
        )

        # Credible interval
        plt.axvline(summ["fb_low"], color=_c("interval", "ci"), linestyle="--", linewidth=1.0, label="95% credible interval")
        plt.axvline(summ["fb_high"], color=_c("interval", "ci"), linestyle="--", linewidth=1.0)
        plt.axvline(summ["fb_best"], color=_c("median", "median"), linestyle="-", linewidth=1.0, label="Median")

        f_b_inv = config.M_bary_known / config.M_total

        # Reference lines
        plt.axvline(
            f_b_MW_inventory,
            color=_c("mw", "mw"),
            linestyle=":",
            linewidth=1.0,
            label=r"$M_{\rm bary}/M_{\rm tot}$ (inventory)",
        )
        if f_b_expected is not None:
            plt.axvline(
                f_b_expected,
                color=_c("expected", "expected"),
                linestyle="--",
                linewidth=2.0,
                label=r"$f_b$=0.08 (expected)",
            )

        plt.xscale("log")
        plt.yscale("log")
        plt.xlabel(r"Baryon fraction $f_b$")
        plt.ylabel("Probability density")
        plt.title(f"Baryon Fraction Posterior Including ISO Contribution ({template} template)")
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            f"results/{figure_name}_{template}.pdf",
            format="pdf",
            bbox_inches="tight",
        )
        plt.show()
        plt.close()

In [ ]:
plot_fb_posterior(results, figure_name="fb_posterior")

## Figure: Robustness of inferred quantities to KDE bandwidth choice

This figure provides a compact summary of the sensitivity of the inferred
quantities to the choice of KDE bandwidth used in the posterior fusion step
(Sect.~\\ref{sec:KDE_fusion}).

For each tested KDE bandwidth factor (`bw_factor`), the figure shows the
central credible interval and median of:

- the joint local ISO mass density $\rho_{\rm local}$, and
- the resulting baryon fraction $f_b$ (for a fixed Galactic template).

Each horizontal bar corresponds to the 95\% credible interval obtained from
the KDE-weighted joint posterior for a given bandwidth factor. The marker
indicates the median of the distribution.

The bandwidth factor `bw_factor = 1.0`, corresponding to Scott’s rule, is
highlighted visually to indicate the default configuration used throughout
the main Results section. Other bandwidth values are shown for comparison
only and are not used elsewhere in the paper.

Expected output:
- Two stacked figures showing credible intervals as horizontal bars,
- One bar per KDE bandwidth factor,
- Medians indicated by point markers.


In [ ]:
def plot_bw_robustness_intervals(
    results,
    figure_prefix="bw_robustness",
    show=True,
):
    """
    Robustness summary for KDE bandwidth choice.

    Produces:
      1) One template-independent figure for rho_local
      2) One figure per template for f_b

    All figures are written to results/ as PDFs.
    """

    # --- helper ---
    def _parse_bw(k):  # "bw_1.0" -> 1.0
        return float(k.split("_", 1)[1])

    bw_keys = sorted(results["bw"].keys(), key=_parse_bw)

    # ============================================================
    # 1) Collect rho_local summaries (template-independent)
    # ============================================================
    rho_rows = []
    for k in bw_keys:
        r = results["bw"][k]["rho_joint_summary"]
        rho_rows.append((k, r["rho_low"], r["rho_best"], r["rho_high"]))

    # --- print rho_local bounds ---
    print("rho_local robustness bounds:")
    print(f"  {'bandwidth':<12}  {'low':>12}  {'best':>12}  {'high':>12}")
    for k, lo, md, hi in rho_rows:
        print(f"  {k:<12}  {lo:>12.4e}  {md:>12.4e}  {hi:>12.4e}")

    # ============================================================
    # 2) Plot rho_local robustness (ONE figure)
    # ============================================================
    plt.figure(figsize=(9, 0.7 + 0.45 * len(bw_keys)))

    for i, (k, lo, md, hi) in enumerate(rho_rows):
        is_default = np.isclose(_parse_bw(k), 1.0)
        col = _c("joint", "ci") if is_default else _c("median", "median")

        plt.hlines(i, lo, hi, color=col, linewidth=3)
        plt.plot(md, i, marker="o", color=col)

    plt.yticks(range(len(bw_keys)), bw_keys)
    plt.xscale("log")
    plt.xlabel(r"$\rho_{\rm local}$ [kg m$^{-3}$]")
    plt.title("Robustness to KDE bandwidth: local ISO mass density")
    plt.tight_layout()

    plt.savefig(
        f"results/{figure_prefix}_rho_local.pdf",
        format="pdf",
        bbox_inches="tight",
    )

    if show:
        plt.show()
    plt.close()



In [ ]:
plot_bw_robustness_intervals(results)

## Tables: CSV exports for the Results section

The following functions generate comma-separated value (CSV) tables containing
numerical summaries of the posterior distributions.

These tables are intended to be:
- imported directly into the manuscript preparation workflow,
- used to populate Tables~\\ref{tab:rho_local_summary} and
  \\ref{tab:MISO_summary} in the paper,
- or archived alongside the notebook for reproducibility.

Each CSV file contains only summary statistics derived from the already-sampled
posteriors; no new calculations are introduced at this stage.

Expected output:
- `rho_local_summary.csv`: joint $\rho_{\rm local}$ intervals for each
  KDE bandwidth,
- `MISO_summary.csv`: Galaxy-integrated ISO mass summaries by template,
- `fb_summary.csv`: baryon fraction summaries by template.


In [ ]:
def write_rho_local_summary_csv(
    results,
    template="baryonic",
    bw_factor=1.0,
    path="results/rho_local_summary.csv",
):
    """
    Write method-level summary of rho_local:
      - O–O population model
      - Flux-based rate inference
      - KDE-weighted joint posterior (bw = 1.0)

    This table is used in the first Results subsection.
    """

    bw_key = f"bw_{bw_factor}"

    tpl = results["templates"][template]
    bw  = results["bw"][bw_key]

    rows = [
        {
            "inference_method": "OO_population_model",
            "rho_low":  tpl["rho_OO_summary"]["rho_low"],
            "rho_med":  tpl["rho_OO_summary"]["rho_best"],
            "rho_high": tpl["rho_OO_summary"]["rho_high"],
        },
        {
            "inference_method": "flux_rate_inference",
            "rho_low":  tpl["rho_rate_summary"]["rho_low"],
            "rho_med":  tpl["rho_rate_summary"]["rho_best"],
            "rho_high": tpl["rho_rate_summary"]["rho_high"],
        },
        {
            "inference_method": "KDE_weighted_joint_posterior",
            "rho_low":  bw["rho_joint_summary"]["rho_low"],
            "rho_med":  bw["rho_joint_summary"]["rho_best"],
            "rho_high": bw["rho_joint_summary"]["rho_high"],
        },
    ]

    df = pd.DataFrame(rows)

    # format for LaTeX tables
    for col in ["rho_low", "rho_med", "rho_high"]:
        df[col] = df[col].map(lambda x: f"{x:.2e}")

    df.to_csv(path, index=False)


In [ ]:
write_rho_local_summary_csv(results)

In [ ]:
def write_MISO_csv(results, path="results/MISO_summary.csv"):
    rows = []
    for bw_key, bw in results["bw"].items():
        for template in results["templates"].keys():
            M = bw[template]["MISO_kde"]
            rows.append({
                "bw_factor": bw_key,
                "template": template,
                "M_low": np.percentile(M, results["meta"]["ci_low"]),
                "M_med": np.percentile(M, 50),
                "M_high": np.percentile(M, results["meta"]["ci_high"]),
            })
    df = pd.DataFrame(rows)
    for col in ["M_low", "M_med", "M_high"]:
        df[col] = df[col].map(lambda x: f"{x:.2e}")

    df.to_csv(path, index=False)


In [ ]:
write_MISO_csv(results)

In [ ]:
def write_fb_csv(results, path="results/fb_summary.csv"):
    rows = []
    for bw_key, bw in results["bw"].items():
        for template in results["templates"].keys():
            fb_kde = bw[template]["f_b_kde_summary"]
            rows.append({
                "bw_factor": bw_key,
                "template": template,
                "fb_low": fb_kde["fb_low"],
                "fb_med": fb_kde["fb_best"],
                "fb_high": fb_kde["fb_high"],
                "f_ISO_low": fb_kde["f_ISO_low"],
                "f_ISO_med": fb_kde["f_ISO_best"],
                "f_ISO_high": fb_kde["f_ISO_high"],
            })
    df = pd.DataFrame(rows)
    for col in ["fb_low", "fb_med", "fb_high", "f_ISO_low", "f_ISO_med", "f_ISO_high"]:
        df[col] = df[col].map(lambda x: f"{x:.2e}")

    df.to_csv(path, index=False)


In [ ]:
write_fb_csv(results)

## Numerical printout

This utility prints all numerical values present in the paper.

The output includes:
- the joint local ISO mass density interval,
- the effective sample size (ESS) of the KDE-weighted posterior,
- Galaxy-integrated ISO mass intervals for each spatial template,
- baryon fraction intervals for each spatial template.

The printout is ordered to follow the structure of the Results section.

Expected output:
- A single, deterministic console block containing all reported values
  for the default KDE bandwidth.


In [ ]:
def print_results(results, bw_factor=1.0):
    bw_key = f"bw_{bw_factor}"

    print("\n=== Local ISO mass density ===")
    r = results["bw"][bw_key]["rho_joint_summary"]
    print(f"rho_local = [{r['rho_low']:.3e}, {r['rho_best']:.3e}, {r['rho_high']:.3e}]")

    w = results["bw"][bw_key]["fusion_weights"]
    ess = 1.0 / np.sum(w**2)
    print(f"Effective sample size (ESS) = {ess:.0f}")

    for template in results["templates"].keys():
        print(f"\n=== Template: {template} ===")

        M = results["bw"][bw_key][template]["MISO_kde"]
        print(
            "M_ISO = "
            f"[{np.percentile(M, results['meta']['ci_low']):.3e}, "
            f"{np.percentile(M, 50):.3e}, "
            f"{np.percentile(M, results['meta']['ci_high']):.3e}] Msun"
        )

        fb = results["bw"][bw_key][template]["f_b_kde_summary"]
        print(
            "f_b = "
            f"[{fb['fb_low']:.3e}, {fb['fb_best']:.3e}, {fb['fb_high']:.3e}]"
        )


In [ ]:
print_results(results=results, bw_factor=1.0
              )

## Sensitivity of the joint posterior to detection efficiency $\epsilon$

The rate-based constraint treats the heliocentric distance at first secure detection as a proxy for the effective survey reach. This implicitly assumes that every ISO passing within $R_{\rm det}$ was detected ($\epsilon = 1$). In practice, heterogeneous survey cadences, limiting magnitudes, and follow-up completeness mean that only a fraction $\epsilon \in (0,1]$ of such objects would be recovered.

Replacing $A_i \to \epsilon\, A_i$ scales the inferred number density by $1/\epsilon$ and shifts $\rho_{\rm local}^{\rm rate}$ accordingly. The O–O prior is unaffected.

This analysis sweeps $\epsilon$ from 0.1 to 1.0, recomputes only the rate channel for each value, and re-fuses with the existing O–O samples to obtain the joint posterior. The resulting figure shows the 95% credible interval and median of $\rho_{\rm local}^{\rm joint}$ as a function of $\epsilon$, with the rate-only and O–O prior intervals shown for reference.

Note: this treats $\epsilon$ as object-independent and assumes the undetected population has the same mass distribution as the detected sample. A power-law mass function would imply many additional low-mass objects below the detection threshold; the net effect on $\rho_{\rm local}$ depends on the slope of the mass function and is deferred to the discussion.

In [ ]:
def compute_rho_rate_samples_with_epsilon(epsilon):
    """
    Compute rate-based posterior samples for a given detection efficiency epsilon.
    Identical to compute_rho_rate_samples() but with a caller-specified epsilon
    instead of config.ISO_detection_completeness.
    """
    t0 = datetime.fromisoformat(config.ISO_first_detection)
    t1 = datetime.fromisoformat(config.ISO_last_detection)
    T_obs = (t1 - t0).days * 86400.0

    R_arr_m = np.array(config.ISO_detection_distance) * config.AU_m
    A_list = np.pi * R_arr_m**2

    lam_samples = gamma.rvs(
        a=config.ISO_N_det + 1, scale=1.0 / T_obs, size=config.N_samples
    )

    masses_arr = np.array(config.ISO_masses)
    idx = np.random.randint(0, len(masses_arr), (config.N_samples, len(masses_arr)))
    mean_mass_samples = masses_arr[idx].mean(axis=1)

    v_inf_m_s = np.array(config.ISO_velocities)
    A_v_sum = np.sum(A_list * v_inf_m_s)

    n0_samples_m3 = lam_samples / (epsilon * A_v_sum)
    rho_rate_samples = mean_mass_samples * n0_samples_m3

    return rho_rate_samples


def epsilon_sensitivity(results, epsilon_values=None, bw_factor=1.0,
                        figure_name="epsilon_sensitivity"):
    """
    Sweep detection efficiency epsilon and show how the joint posterior
    on rho_local responds.

    The O–O samples are reused from the existing run. Only the rate
    channel is recomputed for each epsilon, and the KDE fusion is redone.

    Parameters
    ----------
    results : dict
        Output of run_joint_OO_plus_rate_analysis().
    epsilon_values : list of float
        Detection efficiencies to test. Default: 10 values from 0.1 to 1.0.
    bw_factor : float
        KDE bandwidth factor (default 1.0).
    figure_name : str
        Filename stem for the saved figure.
    """
    if epsilon_values is None:
        epsilon_values = [0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.85, 1.00]

    # Reuse O–O samples from the existing run (any template has the same ones)
    first_template = list(results["templates"].keys())[0]
    rho_OO_samples = results["templates"][first_template]["rho_OO_samples"]

    rng = np.random.default_rng(42)

    # Storage
    medians = []
    lows = []
    highs = []
    ess_values = []

    for eps in epsilon_values:
        # Recompute rate channel for this epsilon
        rho_rate_eps = compute_rho_rate_samples_with_epsilon(eps)

        # Fuse with the same O–O samples
        rho_joint_eps, weights_eps = fuse_OO_and_rate_posteriors_KDE(
            rho_OO_samples=rho_OO_samples,
            rho_rate_samples=rho_rate_eps,
            bw_factor=bw_factor,
            rng=rng,
        )

        lo = np.percentile(rho_joint_eps, config.ci_low)
        med = np.percentile(rho_joint_eps, 50.0)
        hi = np.percentile(rho_joint_eps, config.ci_high)
        ess = 1.0 / np.sum(weights_eps[weights_eps > 0] ** 2)

        lows.append(lo)
        medians.append(med)
        highs.append(hi)
        ess_values.append(ess)

        print(f"  epsilon={eps:.2f}  ->  rho_local = [{lo:.2e}, {med:.2e}, {hi:.2e}]  ESS={ess:.0f}")

    eps_arr = np.array(epsilon_values)
    lows = np.array(lows)
    medians = np.array(medians)
    highs = np.array(highs)

    # --- Also show rate-only 95% CI for reference ---
    rate_lows = []
    rate_highs = []
    for eps in epsilon_values:
        rho_rate_eps = compute_rho_rate_samples_with_epsilon(eps)
        rate_lows.append(np.percentile(rho_rate_eps, config.ci_low))
        rate_highs.append(np.percentile(rho_rate_eps, config.ci_high))
    rate_lows = np.array(rate_lows)
    rate_highs = np.array(rate_highs)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(7, 4.5))

    # Rate-only CI band (background)
    ax.fill_between(eps_arr, rate_lows, rate_highs,
                    color=_c("rate_interval", "rate"), alpha=0.15,
                    label="95% CI (rate-only)")

    # Joint CI band
    ax.fill_between(eps_arr, lows, highs,
                    color=_c("joint_pdf", "ci"), alpha=0.35,
                    label="95% CI (joint)")

    # Joint median
    ax.plot(eps_arr, medians, "-o", color=_c("median", "median"),
            markersize=4, linewidth=1.5, label="Median (joint)")

    # O–O prior 95% CI as horizontal band
    oo_lo = np.percentile(rho_OO_samples, config.ci_low)
    oo_hi = np.percentile(rho_OO_samples, config.ci_high)
    ax.axhspan(oo_lo, oo_hi, color=_c("oo_pdf", "hist"), alpha=0.10,
               label=r"95% CI ($\overline{\rm O}$–O prior)")

    ax.set_yscale("log")
    ax.set_xlabel(r"Detection efficiency $\epsilon$", fontsize=config.fontsize)
    ax.set_ylabel(r"$\rho_{\rm local}$ [kg m$^{-3}$]", fontsize=config.fontsize)
    ax.set_title(
        r"Sensitivity of $\rho_{\rm local}$ to detection efficiency $\epsilon$",
        fontsize=config.fontsize,
    )
    ax.set_xlim(eps_arr.min() - 0.02, eps_arr.max() + 0.02)
    ax.legend(fontsize=10, loc="upper right")
    ax.tick_params(labelsize=11)
    fig.tight_layout()

    fig.savefig(f"results/{figure_name}.pdf", format="pdf", bbox_inches="tight")
    plt.show()
    plt.close()

    return {
        "epsilon": eps_arr,
        "rho_low": lows,
        "rho_median": medians,
        "rho_high": highs,
        "ess": np.array(ess_values),
    }


# --- Run it ---
print("Running epsilon sensitivity analysis...")
eps_results = epsilon_sensitivity(results, bw_factor=1.0)